# SETTING UP THE EVIRONMENT

In [0]:
import json
kafka_connection_json=dbutils.secrets.get(scope="finguard-scope",key="kafka_connection_details")
kafka_config=json.loads(kafka_connection_json)

bootstrap_servers=kafka_config['bootstrap_servers']
api_key=kafka_config['api_key']
api_secret=kafka_config['api_secret']
topic=kafka_config['topic']

In [0]:
jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

# READING BATCH DATA

In [0]:
sample_batch=(spark.read.format("kafka")
                .option("kafka.bootstrap.servers", bootstrap_servers)
                .option("subscribe", topic)
                .option("kafka.security.protocol", "SASL_SSL")
                .option("kafka.sasl.mechanism", "PLAIN")
                .option("kafka.sasl.jaas.config", jaas_config)
                .option("startingOffsets", "earliest")
                .load()
)

In [0]:
sample_batch.count()

In [0]:
display(sample_batch)

# CHANGE DATA TYPE OF  KEY & VALUE TO STRING

In [0]:
from pyspark.sql.functions import col
parsed_batch = sample_batch.select(
    col("key").cast("string"),
    col("value").cast("string").alias("value_string") ,
    col("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")

)

In [0]:
display(parsed_batch)

# WRITE THE DATA TO BRONZE TABLE

In [0]:
parsed_batch.write.saveAsTable("finguard.bronze.transactions_batch_test")

In [0]:
display(finguard.bronze.transactions_batch_test)

# READING STREAM DATA

In [0]:
streaming_df=(spark.readStream.format("kafka")
                .option("kafka.bootstrap.servers", bootstrap_servers)
                .option("subscribe", topic)
                .option("kafka.security.protocol", "SASL_SSL")
                .option("kafka.sasl.mechanism", "PLAIN")
                .option("kafka.sasl.jaas.config", jaas_config)
                .option("startingOffsets", "earliest")
                .load()
)

# CHANGE DATA TYPE OF  KEY & VALUE TO STRING

In [0]:
from pyspark.sql.functions import col
parsed_streaming_df = streaming_df.select(
    col("key").cast("string"),
    col("value").cast("string").alias("value_string") ,
    col("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")

)

# WRITE STRAMING DATA TO THE TABLE

In [0]:
streaming_query=(parsed_streaming_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointlocation", "/Volumes/finguard/source/transactions/checkpoint/")
.trigger(availableNow=True)
.toTable("finguard.bronze.transactions_streaming_test")
)

print("Query Started with query id: ", streaming_query.id)

In [0]:
%sql
select * from finguard.bronze.transactions_streaming_test